## 0. Kernel setup (run in a terminal, not in this notebook)

Before launching this notebook, create and select a conda environment kernel (`2ndWorkshop`).

### Purdue Gilbreth cluster

```bash
module load conda
conda-env-mod create -n ENV_NAME_HERE -j
module use $HOME/privatemodules
module load conda-env/ENV_NAME_HERE-py3.10.11
```

Replace `ENV_NAME_HERE` with your environment name (`2ndWorkshop`), then select the matching kernel in Jupyter before running the cells below.

Check the env and kernel were created:
```bash
conda env list
jupyter kernelspec list
```

### Local machine (conda)

```bash
conda create -n 2ndWorkshop python=3.10 pip -y
conda activate 2ndWorkshop
pip install ipykernel
python -m ipykernel install --user --name 2ndWorkshop --display-name "Python (2ndWorkshop)"
```

Select the `Python (2ndWorkshop)` kernel, run the install cell below once, then restart the kernel.

**Prerequisite:** `Workshop2_Part2a_Server.ipynb` must already be running (in a separate
kernel) before you run the cells below.

**Note:** This notebook needs an LLM backend that supports tool calling — Ollama running
locally is the free default (`ollama pull llama3.2`), or set a Purdue GenAI / OpenAI key.
See the README for details.


# Workshop 2, Part 2b: LangGraph Agent (Boilermaker TA)

This is the **agent half** of Part 2. It connects to the MCP tool server from
`Workshop2_Part2a_Server.ipynb` over HTTP, discovers its tools, and orchestrates them in a
**LangGraph** agent loop to complete a task — that's the Boilermaker TA.

**Before running this notebook:** start `Workshop2_Part2a_Server.ipynb` in a separate
kernel and leave it running.

**What you'll do:**
- Connect to the MCP server and discover its tools
- Build a LangGraph agent graph (agent node + tool node, looping until done)
- Run the Boilermaker TA on a real task and inspect the tool calls it makes

**LLM backend:** same default as Workshop 1 and Part 1 — a small local Hugging Face
model, no API key or external server required. Tool-calling reliability drops with model
size, though, so **C2** also shows how to switch to Ollama, OpenAI, or Anthropic with one
line if the local model struggles.


## 0. Install dependencies

Run once, then restart the kernel.

In [1]:
! pip install torch transformers accelerate langchain-huggingface langchain langchain-core langgraph langchain-mcp-adapters python-dotenv


---
# Part C: LangGraph Agent

The agent is a **stateful graph** with two nodes:
- `agent` — the LLM, decides whether to call a tool or return a final answer
- `tools` — a `ToolNode` that executes whatever tool the LLM requested

The `should_continue` router inspects the last message: if it has `tool_calls`, go to `tools`; otherwise end.

Make sure `Workshop2_Part2a_Server.ipynb` is running in another kernel before you run the
cells below.


## C1. Imports for the agent

In [2]:
import os
import asyncio
import logging
from pathlib import Path
from typing import Annotated
from typing_extensions import TypedDict

from dotenv import load_dotenv
from transformers import pipeline
from langchain.chat_models import init_chat_model
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

load_dotenv()

BASE_DIR = Path(".").resolve()
ANNOUNCEMENTS_FILE = BASE_DIR / "workshop_outputs/announcements.txt"   # written by the create_notification tool

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


/Users/elhambarezi/miniconda3/envs/2ndWorkshop/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## C2. Configure the LLM backend

Everything is set with plain variables in the next cell — no environment variables needed
(API keys still go in `.env`). Three presets are included; uncomment the one you want and
re-run the cell.

**Tradeoff to know:** tool-calling reliability scales with model size. Qwen2.5-Instruct
models support function calling even at 0.5B, but a 0.5B model will miss or malform tool
calls more often than a hosted API. If the agent seems to ignore its tools (e.g. it never
writes to `announcements.txt`), switch away from the local preset.

**Where you're running matters:**
- **Your own machine (e.g. your Mac):** the local HF model or Ollama both work out of the
  box (Ollama needs its server running — `ollama serve`, or already running in the
  background if you installed the app — and the model pulled first, `ollama pull llama3.2`).
- **Purdue Gilbreth (remote GPU workshop):** Ollama isn't preinstalled, but it works fine
  with a one-time no-sudo userspace install (it's a static binary that only needs a local
  port, no root required):
  ```bash
  curl -L https://ollama.com/download/ollama-linux-amd64.tgz -o ollama-linux-amd64.tgz
  mkdir -p ~/.local/bin
  tar -C ~/.local/bin -xzf ollama-linux-amd64.tgz
  echo 'export PATH="$HOME/.local/bin:$PATH"' >> ~/.bashrc && source ~/.bashrc

  # Home directory quota is usually small -- point model storage at scratch/depot instead:
  export OLLAMA_MODELS="/path/to/your/large/storage/directory"

  ollama serve            # keep this running in its own terminal for the whole session
  ollama pull llama3.2     # in a second terminal, once `ollama serve` is up
  ```
  Run this in a terminal within the same Gilbreth job/session as your Jupyter kernel (so
  `ollama serve` and the notebook share the same node), then use the Ollama preset below
  exactly as you would locally. If the download step can't reach the internet from your
  allocated node, do it on the login node first and skip straight to `ollama serve` once
  the binary and models are in place. Otherwise, **Purdue GenAI** below needs zero server
  management and is the easiest option on Gilbreth.

### Purdue GenAI Studio (`genai.rcac.purdue.edu`)

Purdue's hosted GenAI Studio exposes an OpenAI-compatible chat endpoint at
`https://genai.rcac.purdue.edu/api/chat/completions`, authenticated with a bearer
token/API key from your GenAI Studio account. `langchain`'s OpenAI-compatible client
posts to `{base_url}/chat/completions` automatically, so:
- `LLM_BASE_URL = "https://genai.rcac.purdue.edu/api"` (note: **no** `/chat/completions` —
  the client appends that itself)
- put your GenAI Studio API key in `.env` as `OPENAI_API_KEY=...`
- `LLM_MODEL = "openai:llama3.1:latest"` (or whatever model name your GenAI Studio account
  lists — model names look like Ollama tags, e.g. `llama3.1:latest`)

**Caveat:** Purdue's own example only demonstrates a plain chat completion — it doesn't
confirm the endpoint forwards OpenAI's `tools`/`tool_choice` parameters to the underlying
model. Test it with C7 below and check whether `announcements.txt` gets written; if the
agent never calls tools through this backend, fall back to Ollama or an Anthropic/OpenAI key.

| Preset | Works on | Notes |
|---|---|---|
| **Local HF (default)** | Mac and Gilbreth | `Qwen/Qwen2.5-0.5B-Instruct`, no server/key needed, weakest tool calling |
| Ollama | Mac out of the box; Gilbreth with a one-time no-sudo userspace install (see above) | free, much better tool calling than local HF |
| Purdue GenAI | Mac and Gilbreth | zero server management; needs an API key from GenAI Studio; tool-calling support unconfirmed (see caveat above) |
| OpenAI / Anthropic | Mac and Gilbreth (if outbound internet allowed) | needs `OPENAI_API_KEY` / `ANTHROPIC_API_KEY` in `.env`; most reliable tool calling |


In [ ]:
# --- Edit these directly, no env vars needed. Uncomment ONE preset. ---

LOCAL_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # used only when LLM_BACKEND == "local"

# 1) Local HF model (default) -- runs anywhere (Mac or Gilbreth), no server or API key,
#    but weakest tool-calling reliability.
LLM_BACKEND  = "local"
LLM_MODEL    = None
LLM_BASE_URL = None

# 2) Ollama -- best free tool-calling. Works out of the box on your Mac. On Gilbreth it
#    needs a one-time no-sudo userspace install with `ollama serve` kept running in a
#    terminal in the same job/session as this notebook (see the C2 markdown above for
#    the install steps) -- once that's running, this preset works the same either way.
# LLM_BACKEND  = "api"
# LLM_MODEL    = "ollama:llama3.2"
# LLM_BASE_URL = None

# 3) Purdue GenAI Studio -- OpenAI-compatible endpoint at genai.rcac.purdue.edu, no
#    locally-running server needed, so this is the zero-setup option on Gilbreth. Put
#    your GenAI Studio API key in `.env` as OPENAI_API_KEY. Note: tool-calling support
#    through this endpoint is unconfirmed -- test with C7 and check announcements.txt.
# LLM_BACKEND  = "api"
# LLM_MODEL    = "openai:llama3.1:latest"                    # or another model from your GenAI Studio account
# LLM_BASE_URL = "https://genai.rcac.purdue.edu/api"          # client appends /chat/completions itself

MCP_SERVER_URL = "http://127.0.0.1:8001/mcp"

print("LLM backend: ", LLM_BACKEND, f"({LOCAL_MODEL})" if LLM_BACKEND == "local" else f"({LLM_MODEL})")
print("MCP server:  ", MCP_SERVER_URL)


def _check_ollama_running(base_url: str = "http://localhost:11434") -> None:
    """Ollama models are served locally -- give a clear error if the server isn't up
    instead of letting the connection failure surface deep inside init_chat_model()."""
    import urllib.request
    try:
        urllib.request.urlopen(base_url, timeout=2)
    except Exception:
        raise RuntimeError(
            f"Could not reach Ollama at {base_url}. Start it with `ollama serve` "
            "(or open the Ollama app) and make sure the model is pulled, e.g. "
            "`ollama pull llama3.2`, then re-run this cell. On Gilbreth, `ollama serve` "
            "must be running in a terminal in the same job/session as this notebook -- "
            "see the C2 markdown cell above for the no-sudo install steps."
        )


def create_llm():
    """
    Local Hugging Face model by default (same as Workshop 1). Set LLM_BACKEND = "api" above to
    route through init_chat_model() instead -- Ollama, Purdue GenAI, OpenAI, Anthropic, or any
    OpenAI-compatible endpoint (LLM_BASE_URL lets you point at Open WebUI, vLLM, LM Studio,
    Purdue GenAI, etc.) -- all without touching this function.
    """
    if LLM_BACKEND == "local":
        text_gen = pipeline("text-generation", model=LOCAL_MODEL, max_new_tokens=512)
        return ChatHuggingFace(llm=HuggingFacePipeline(pipeline=text_gen))

    if LLM_MODEL.startswith("ollama:"):
        _check_ollama_running(LLM_BASE_URL or "http://localhost:11434")

    kwargs = {"temperature": 0}
    if LLM_BASE_URL:
        kwargs["base_url"] = LLM_BASE_URL
    return init_chat_model(LLM_MODEL, **kwargs)


## C3. AgentState — the shared message history

In [4]:
class AgentState(TypedDict):
    # add_messages is a reducer: new messages are appended, not replaced
    messages: Annotated[list, add_messages]

## C4. Build the LangGraph agent graph

```
START → agent ──┬─(tool_calls?)→ tools → agent (loop)
                └─(no tool calls)→ END
```

In [5]:
def build_graph(tools):
    llm            = create_llm()
    # bind_tools attaches the tool schemas (name, args, docstring) to every LLM call so
    # the model can choose to emit a tool_calls request instead of plain text.
    llm_with_tools = llm.bind_tools(tools)

    def agent_node(state: AgentState) -> dict:
        # The model sees the full message history (including prior tool results) and
        # decides whether it needs another tool call or can answer directly.
        response = llm_with_tools.invoke(state["messages"])
        return {"messages": [response]}

    def should_continue(state: AgentState) -> str:
        # Router: if the last message has tool_calls, execute them; otherwise finish
        last_message = state["messages"][-1]
        if last_message.tool_calls:
            return "tools"
        return END

    # ToolNode is a prebuilt LangGraph node: it reads tool_calls off the last message,
    # runs the matching tool by name, and appends the results as ToolMessages.
    tool_node = ToolNode(tools)

    graph = StateGraph(AgentState)
    graph.add_node("agent", agent_node)
    graph.add_node("tools", tool_node)
    graph.add_edge(START, "agent")
    graph.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
    graph.add_edge("tools", "agent")   # after tool execution, go back to agent
    return graph.compile()


## C5. System prompt for the Boilermaker TA

In [6]:
SYSTEM_PROMPT = """
You are the Boilermaker Autonomous TA for a Purdue course.
Your job is to help students and instructors manage course planning using the tools available.

Tools:
- search_knowledge_base(query): search the course knowledge base for syllabus details, policies, and study guidance.
- get_academic_calendar(query): read academic calendar events.
- create_notification(subject, body): write a formatted announcement to the announcements file.

Always use the tools when you need facts from the knowledge base or calendar.
Use create_notification only when you have a final announcement ready to write.
"""

## C6. Connect to the MCP server and run the agent

In [ ]:
DEFAULT_QUESTION = (
    "A professor wants to schedule a CS course review session that does not conflict "
    "with holidays or exams. Summarize the plan and write a notification announcement for students."
)


async def run_agent(question: str = DEFAULT_QUESTION):
    log.info("Connecting to MCP server at %s", MCP_SERVER_URL)

    try:
        client = MultiServerMCPClient({
            "boiler_ta": {
                "url": MCP_SERVER_URL,
                "transport": "streamable_http",
            },
        })
        tools = await client.get_tools()
    except Exception as e:
        log.error("Could not connect to MCP server: %s", e)
        return

    log.info("Loaded %d tools: %s", len(tools), [t.name for t in tools])

    agent  = build_graph(tools)
    result = await agent.ainvoke({
        "messages": [SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=question)],
    })

    for msg in result["messages"]:
        if isinstance(msg, AIMessage) and msg.content:
            print("\nAgent reply:\n", msg.content)


# Run the async agent inside the notebook
await run_agent()

13:11:13  INFO      Connecting to MCP server at http://127.0.0.1:8001/mcp
13:11:13  INFO      HTTP Request: POST http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
13:11:13  INFO      Received session ID: 5aca52addbd64b31b3c2e88390ac88fd


## C7. Try a custom question

In [ ]:
await run_agent("What assignment deadlines are coming up in the next two weeks? Summarize them for students.")

12:52:52  INFO      Connecting to MCP server at http://127.0.0.1:8001/mcp
12:52:52  INFO      HTTP Request: POST http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
12:52:52  INFO      Received session ID: 32b1a253c4324325b598cbffbf0acf06
12:52:52  INFO      Negotiated protocol version: 2025-11-25
12:52:52  INFO      HTTP Request: POST http://127.0.0.1:8001/mcp "HTTP/1.1 202 Accepted"
12:52:52  INFO      HTTP Request: GET http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
12:52:52  INFO      HTTP Request: POST http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
12:52:52  INFO      HTTP Request: DELETE http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
12:52:52  INFO      Loaded 4 tools: ['search_knowledge_base', 'retrieve_docs', 'get_academic_calendar', 'create_notification']
12:52:52  INFO      HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
12:52:52  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwe


Agent reply:
 <|im_start|>system

You are the Boilermaker Autonomous TA for a Purdue course.
Your job is to help students and instructors manage course planning using the tools available.

Tools:
- search_knowledge_base(query): search the course knowledge base for syllabus details, policies, and study guidance.
- get_academic_calendar(query): read academic calendar events.
- create_notification(subject, body): write a formatted announcement to the announcements file.

Always use the tools when you need facts from the knowledge base or calendar.
Use create_notification only when you have a final announcement ready to write.
<|im_end|>
<|im_start|>user
What assignment deadlines are coming up in the next two weeks? Summarize them for students.<|im_end|>
<|im_start|>assistant
I'm sorry, but I don't have access to real-time information about upcoming assignments or deadlines. However, typically, assignments might include due dates that are communicated to students via email or through an a

## C8. Check the announcements file

In [ ]:
if ANNOUNCEMENTS_FILE.exists():
    print(ANNOUNCEMENTS_FILE.read_text())
else:
    print("No announcements written yet.")

No announcements written yet.


---
## Extension ideas

- Swap `LLM_MODEL` in `.env` to use a different backend (OpenAI, Purdue GenAI, vLLM)  
- Add a new `@app.tool` in `Workshop2_Part2a_Server.ipynb` (e.g. `get_grades`, `send_email`), restart that notebook's kernel, then re-run the agent here  
- Point `MCP_SERVER_URL` at a server running on another machine or port  
- Replace the FAISS index (in the server notebook) with a persistent vector database (ChromaDB, Qdrant)  
- Connect the LoRA-fine-tuned model from Workshop 1 as the LLM backend
